In [14]:
from dataclasses import dataclass 
import numpy as np
import matplotlib.pyplot as plt

In [4]:
@dataclass
class Hydrology:
    A: float  = 1 # Area of the catchment (m^2)
    B: float  = 1 # Seepage coefficient (m/s)

    P: float  = 1 # Precipitation rate (m/s)
    E: float  = 0.5 # Evaporation rate (m/s)
    Phi: float  = 1 # Percentage pumped out

    alpha: float  = 0.5 # Shape scaling exponent
    beta: float  = 0.5 # Shape scaling exponent
    gamma: float  = 0.5 # Shape scaling exponent

    sigma_1: float  = 0.1 # Standard deviation of the first noise term
    sigma_2: float  = 0.1 # Standard deviation of the second noise term
    sigma_3: float  = 0.1 # Standard deviation of the third noise term

    # For Psi
    c: float  = 1 # Shape scaling exponent
    e: float  = 1 # Shape scaling exponent
    a_p: float  = 1 # Shape scaling exponent
    b_p: float  = 1 # Shape scaling exponent
    n: float  = 1 # Shape scaling exponent

$$\psi(X) = c + e\left(\frac{X}{a_p}\right)^{\frac{n}{b_p}}$$

In [5]:
def psi(X, p: Hydrology):
    """
    This represents the ratio of the surface area of the surrounding land (catchment) to the surface 
    area of the reservoir itself.
    """
    return (p.c + p.e * (X / p.a_p) ** (p.n / p.b_p))

$$\psi'(X) = \frac{n \cdot e}{b_p \cdot a_p^{n/b_p}} X^{\frac{n-b_p}{b_p}}$$

In [6]:
def psi_prime(X, p: Hydrology):
    return (p.e * (p.n / p.b_p) * (1 / p.a_p) ** (p.n / p.b_p) * X ** ((p.n / p.b_p) - 1))

$$F(x) = A x^\alpha \Big[ P(1 - \Phi) + \psi(x)(P\Phi - E) \Big] - Ax^\beta - Bx^\gamma$$
$$F_1 = P(1 - \Phi) + \psi(X)(P\Phi - E)$$

$$F_2 = -A X^\beta - B X^\gamma$$

$$F(X) = A X^\alpha F_1 + F_2$$

In [8]:
def F(X, p: Hydrology):
    psi_x = psi(X, p)
    F1 = p.P * (1 - p.Phi) + psi_x * (p.P * p.Phi - p.E)
    F2 = -p.A * X**p.beta - p.B * X**p.gamma
    return (p.A*X**p.alpha * F1 + F2)

The Diffusion Terms:
1. $$g_1(X) = A \sigma_1 X^\alpha (1 - \Phi + \Phi\psi(X))$$
2. $$g_2(X) = -A \sigma_2 X^\alpha \psi(X)$$
3. $$g_3(X) = -A \sigma_3 P X^\alpha (1 - \psi(X))$$

In [9]:
def Calculate_g(X, p: Hydrology):
    psi_x = psi(X, p)
    g1 = p.A * p.sigma_1 * X**p.alpha * (1 - p.Phi + psi_x * p.Phi)
    g2 = - p.A * p.sigma_2 * X**p.alpha * psi_x
    g3 = - p.A * p.sigma_3 *p.P * X**p.alpha * (1 - psi_x) 
    return [g1, g2, g3]

The Derivatives:
1. $$g_1'(X) = A \sigma_1 \left[ \alpha X^{\alpha-1} (1 - \Phi + \Phi\psi(X)) + X^\alpha \Phi \psi'(X) \right]$$
2. $$g_2'(X) = -A \sigma_2 \left[ \alpha X^{\alpha-1} \psi(X) + X^\alpha \psi'(X) \right]$$
3. $$g_3'(X) = -A \sigma_3 P \left[ \alpha X^{\alpha-1} (1 - \psi(X)) - X^\alpha \psi'(X) \right]$$

In [11]:
def Calculate_g_prime(X, p: Hydrology):
    psi_x = psi(X, p)
    psi_prime_x = psi_prime(X, p)
    g1_prime = p.A * p.sigma_1 * (p.alpha * X**(p.alpha - 1) * (1 - p.Phi + psi_x * p.Phi) + X**p.alpha * psi_prime_x * p.Phi)
    g2_prime = - p.A * p.sigma_2 * (p.alpha * X**(p.alpha - 1) * psi_x + X**p.alpha * psi_prime_x)
    g3_prime = - p.A * p.sigma_3 * p.P * ((p.alpha * X**(p.alpha - 1) * (1 - psi_x) - X**p.alpha * psi_prime_x))
    return [g1_prime, g2_prime, g3_prime]

In standard ODEs, the drift is just $F(X)$. But because we are converting a Stratonovich SDE to an Itô SDE for our numerical solver, we have to add the noise-induced drift correction (also called the Stratonovich correction).The formula for the total Itô drift is:
$$\text{Drift}(X) = F(X) + \frac{1}{2} \sum_{i=1}^{3} g_i(X)g_i'(X)$$

In [13]:
def Drift(X, p: Hydrology):
    g_vals = Calculate_g(X, p)
    g_prime_vals = Calculate_g_prime(X, p)
    corrected_term = (1/2) * sum(g_val * g_prime_val for g_val, g_prime_val in zip(g_vals, g_prime_vals))
    return F(X, p) + corrected_term